# Cleaning downloaded Data from GISAID

Author: Alexander Maksiaev

Purpose: Download and clean GISAID data, after de-duplicating from Andersen/NCBI Virus data

Notes: 
* The "downloads" folder MUST be your computer's downloads folder, or wherever your browser automatically downloads files. This folder must also be cleaned in between each run of this code.
* The returned files from this code will be stored in a separate folder after running -- no other action is needed, aside from cleaning the original downloads folder after this code runs.  
* This file MUST be in the same folder as "utils.py"

## Housekeeping ##

In [1]:
import os
import shutil
import pandas as pd
import numpy as np
import dateutil
import openpyxl
from itertools import islice
from pathlib import Path
import importlib
import utils  
importlib.reload(utils)
from utils import * 

pd.options.mode.chained_assignment = None # suppress warnings when using slices to make new columns

### Inputs and Paths

In [2]:
# Paths

home = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/"
references = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu/references"
downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
andersen_ncbi_virus_gisaid = home + "Combinations/NCBI_Virus_Andersen_GISAID/" 

# Collect user input

locations = "Antarctica,North America,South America"
genotypes = ["A3"] 
start_date = "2021-11-01"
end_date = "2026-08-07"
date_range = dateutil.parser.parse(start_date).strftime("%m-%d-%Y") + "--" + dateutil.parser.parse(end_date).strftime("%m-%d-%Y")

os.chdir(downloads)

# Create directories if needed
downloads_saved = home + "GISAID/downloads/" + start_date + "--" + end_date + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
andersen_ncbi_virus = home + "NCBI_Virus/complete/" + dateutil.parser.parse(start_date).strftime("%m-%d-%Y") + "--" + dateutil.parser.parse(end_date).strftime("%m-%d-%Y") + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
# andersen_ncbi_virus = home + "Combinations/NCBI_Virus_Andersen/" + dateutil.parser.parse(start_date).strftime("%m-%d-%Y") + "--" + dateutil.parser.parse(end_date).strftime("%m-%d-%Y") + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
gisaid_files = home + "GISAID/complete/" + start_date + "--" + end_date + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
complete_files = andersen_ncbi_virus_gisaid + dateutil.parser.parse(start_date).strftime("%m-%d-%Y") + "--" + dateutil.parser.parse(end_date).strftime("%m-%d-%Y") + "_" + locations.replace(",", "_").replace(" ", "_") + "/"

if not os.path.exists(gisaid_files): # checking if the directory exists or not
    os.makedirs(gisaid_files) # if the directory is not present then create it

if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

# Get list of genotypes and states

os.chdir(references)

states = pd.read_csv("states_ref.csv")



## Download all files, convert fasta files to dataframes

In [3]:
all_metadata_files = []
all_fasta_files = []

if not os.path.exists(downloads_saved): # checking if the directory exists or not
    os.makedirs(downloads_saved) # if the directory is not present then create it

# Move downloaded files to saved downloads
for dirpath, dirs, files in os.walk(downloads):
    if len(files) > 0: # If we have any files that need to be moved
        for file in files:
            file_name = os.path.join(dirpath, file)
            destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
            try:
                shutil.move(file_name, destination_path)
            except:
                print("Error moving file", file_name)
                continue 
    else: # If we have downloaded files saved already
        continue
    break 

for dirpath, dirs, files in os.walk(downloads_saved):
    for file in files:
        file_name = os.path.join(dirpath, file)

        print(file_name)

        # Now go through files and get contents
        if ".xls" in file_name:
            metadata = pd.read_excel(file_name)
            all_metadata_files.append(metadata)
        if ".fasta" in file_name:
            fasta_file = fasta_df(file_name, states) # Convert fasta file to dataframe
            all_fasta_files.append(fasta_file)
    break 

C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/GISAID/downloads/2021-11-01--2026-08-07_Antarctica_North_America_South_America/gisaid_epiflu_isolates.xls
C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/GISAID/downloads/2021-11-01--2026-08-07_Antarctica_North_America_South_America/gisaid_epiflu_sequence.fasta


In [4]:
# Concatenate metadata

metadata_concat = pd.DataFrame()
for metadata_file in all_metadata_files:
    metadata_concat = pd.concat([metadata_concat, metadata_file])

In [5]:
# Function to get metadata
def separate_fasta_by_segments(metadata, fasta, animals_df, genotypes): #, b313_fasta, d11_fasta):

    fasta = fix_animals(fasta, animals_df) # Fix animals first

    unique_segments = list(set(fasta["Segment"])) # Get list of segments

    # “>EPI_ID|Isolate_name|subtype|collection_date|host_type|genotype”

    segment_fastas = [] # Get a list of fastas, separated by segment
    for genotype in genotypes: # .keys(): # For each genotype
        for seg in unique_segments: # For each segment
            print(list(set(metadata["Genotype"])))
            xls = metadata[(metadata["Genotype"].str.contains(genotype)) & metadata["Publishing_Embargo_Until"].isna()] # [metadata["Genotype"].apply(lambda x: x.split(" ")[0]) == genotype] # Get only the metadata corresponding to that genotype
            xls = xls.rename(columns={"Isolate_Id":"Identifier"})

            fasta_seg_pre = fasta.merge(xls, how="right", on="Identifier")

            fasta_seg = fasta_seg_pre[fasta_seg_pre["Segment"] == seg]

            fasta_seg["Genotype"] = genotype

            # Rename sequences 
            new_name = ">" + fasta_seg["Identifier"] + "|" + fasta_seg["Isolate_Name_x"] + "|" + fasta_seg["Subtype_x"] + "|" + fasta_seg["Geo_Location"] + "|" + fasta_seg["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else x) + "|" + fasta_seg["Host_Type"] + "|" + fasta_seg["Genotype"] #.apply(lambda x: "" if x != "human" else "|human")
            fasta_seg["full_header"] = new_name
            fasta_seg = fasta_seg.rename(columns={"Isolate_Name_x":"Isolate_Name"})
            # print(fasta_seg["New_Name"])

            segment_fastas.append(fasta_seg)
            print(fasta_seg[["full_header", "Header", "Isolate_Id", "Isolate_Name", "Subtype_x", "Segment", "Geo_Location", "Date Collected", "Identifier", "Host_Type", "Isolate_Name_y", "Subtype_y", "Genotype", "Location", "Collection_Date"]])


    return segment_fastas, unique_segments

In [6]:
# Separate fastas by segment -- results in number of downloaded fastas * number of genotypes * 8 segments
segment_fastas = []
unique_animals_all = []
for i, fasta in enumerate(all_fasta_files):

    metadata = metadata_concat

    unique_animals = sort_animals(fasta) # Find unique animals
    # print("Animals: ", unique_animals)
    unique_animals_all.append(unique_animals)

    os.chdir(references)
    animals_ref = pd.read_csv("animals_ref.csv")

    fastas, unique_segments = separate_fasta_by_segments(metadata, fasta, animals_ref, genotypes) # Separate the fasta dataframes into 8 different files based on segment

    for fasta in fastas:
        segment_fastas.append(fasta)

["A3 (<i style='font-size:11px'>GenoFLU</i>) / EA-2020-C (<i style='font-size:11px'>Genin</i>) / G.4 (<i style='font-size:11px'>ggFLU</i>)", "A3 (<i style='font-size:11px'>GenoFLU</i>) / unassigned (<i style='font-size:11px'>Genin</i>) / G.6 (<i style='font-size:11px'>ggFLU</i>)", "A3 (<i style='font-size:11px'>GenoFLU</i>) / EA-2020-C (<i style='font-size:11px'>Genin</i>)", "A3 (<i style='font-size:11px'>GenoFLU</i>) / unassigned (<i style='font-size:11px'>Genin</i>) / B.23.2 (<i style='font-size:11px'>ggFLU</i>)", "A3 (<i style='font-size:11px'>GenoFLU</i>) / EA-2020-C (<i style='font-size:11px'>Genin</i>) / B.23.2 (<i style='font-size:11px'>ggFLU</i>)", "A3 (<i style='font-size:11px'>GenoFLU</i>) / EA-2020-C (<i style='font-size:11px'>Genin</i>) / G.6 (<i style='font-size:11px'>ggFLU</i>)", "A3 (<i style='font-size:11px'>GenoFLU</i>) / EA-2020-C (<i style='font-size:11px'>Genin</i>) / G.8 (<i style='font-size:11px'>ggFLU</i>)", "A3 (<i style='font-size:11px'>GenoFLU</i>) / EA-2020-C

In [7]:
print(len(segment_fastas))

8


## De-Duplication

In [8]:
# Get files from Andersen and NCBI Virus

# Grab files
andersen_ncbi = {} # Results in number of genotypes * 8 segments
for dirpath, dirs, files in os.walk(andersen_ncbi_virus):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if ".fasta" in file_name:
        # print(file_name)
            segment_genotype = "_".join(file_name.split("/")[-1].split("_")[0:2])
            fasta_file = fasta_df_complete(file_name, states) # Convert fasta file to dataframe
            fasta_file["full_header"] = fasta_file["Header"].apply(lambda x: ">" + x)
            andersen_ncbi[segment_genotype] = fasta_file
    break 

In [9]:
print(andersen_ncbi)

{'A3_HA':                                                 Header  \
0    SRR38343085|A/Common_Eider/AK/26G06093-001-ori...   
1    SRR38343082|A/Great_Blue_Heron/WA/26G06092-001...   
2    SRR38343081|A/Great_Blue_Heron/WA/26G06092-002...   
3    SRR38343080|A/Great_Blue_Heron/WA/26G06092-003...   
4    SRR38563565|A/Herring_Gull/CA/26G06613-005-ori...   
..                                                 ...   
417  GCA_039823905.1|A/northern_pintail/USA/IZ22_08...   
418  GCA_039640125.1|A/black-legged_kittiwake/Alask...   
419  GCA_039320585.1|A/Northern_Pintail/USA/IZ22_03...   
420  GCA_039295575.1|A/Bald_Eagle/Alaska/22-013001-...   
421  GCA_039296015.1|A/Bald_Eagle/Alaska/22-013831-...   

                 Isolate_Id  \
0     26G06093-001-original   
1     26G06092-001-original   
2     26G06092-002-original   
3     26G06092-003-original   
4     26G06613-005-original   
..                      ...   
417               IZ22_0812   
418  23-029675-001-original   
419          I

### Collect partial isolates from all parties

In [10]:
# Do all segments, not just HA 

# Check isolate IDs to see if they already exist in Andersen/NCBI
# Match based on year AND partial isolate ID, as some partials may be identical between years

# GISAID partial isolates
gisaid_list = {}
for gisaid_fasta in segment_fastas:
    if len(gisaid_fasta) > 0:

        gisaid_fasta["Partials"] = gisaid_fasta["Isolate_Id"].apply(partial_isolate)
        gisaid_fasta["Year"] = gisaid_fasta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x).year))
        genotype_gisaid_fasta = gisaid_fasta.drop_duplicates(subset=["Partials", "Year", "Segment"], keep="first")
        if genotype_gisaid_fasta["Genotype"].values[0].split(" ")[0] not in gisaid_list.keys():
            gisaid_list[genotype_gisaid_fasta["Genotype"].values[0].split(" ")[0] + "_" + genotype_gisaid_fasta["Segment"].values[0]] = genotype_gisaid_fasta
        else:
            gisaid_list[genotype_gisaid_fasta["Genotype"].values[0].split(" ")[0] + "_" + genotype_gisaid_fasta["Segment"].values[0]] = pd.concat([gisaid_list[genotype_gisaid_fasta["Genotype"].values[0] + "_" + genotype_gisaid_fasta["Segment"].values[0]], genotype_gisaid_fasta]).drop_duplicates(subset=["Partials", "Year", "Segment"], keep="last")
        
        # print(genotype_gisaid_fasta)
        
# NCBI_Virus/Andersen partial isolates    
andersen_ncbi_genotypes = {} # Results in # of genotypes
for key in andersen_ncbi:
    print(key)
    andersen_ncbi_fasta = andersen_ncbi[key] 

    # Find partial Isolate IDs -- humans and non-humans have different locations for isolates
    andersen_ncbi_fasta_nonhuman = andersen_ncbi_fasta[andersen_ncbi_fasta["Host_Type"] != "human"]
    andersen_ncbi_fasta_nonhuman["Partials"] = andersen_ncbi_fasta_nonhuman["Isolate_Id"].apply(partial_isolate) # For non-humans, apply partial function to isolate ids

    andersen_ncbi_fasta_human = andersen_ncbi_fasta[andersen_ncbi_fasta["Host_Type"] == "human"]
    andersen_ncbi_fasta_human["Partials"] = andersen_ncbi_fasta_human["Isolate_Id"].apply(partial_isolate)

    # Concatenate humans and non-humans
    andersen_ncbi_fasta = pd.concat([andersen_ncbi_fasta_human, andersen_ncbi_fasta_nonhuman])

    # Get year and segment
    andersen_ncbi_fasta["Year"] = andersen_ncbi_fasta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if len(x) > 0 else x)
    andersen_ncbi_fasta["Segment"] = key.split("_")[-1]

    if andersen_ncbi_fasta["Genotype"].values[0].split(" ")[0] not in andersen_ncbi_genotypes.keys(): # If we haven't already seen this genotype
        andersen_ncbi_genotypes[andersen_ncbi_fasta["Genotype"].values[0].split(" ")[0].split("_")[0] + "_" + andersen_ncbi_fasta["Segment"].values[0]] = andersen_ncbi_fasta # Add fasta to genotype (To include "Not" assigned) dictionary



A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2


In [11]:
for gisaid_df in gisaid_list:
    print(gisaid_df)
    print(gisaid_list[gisaid_df])
    print(len(gisaid_df))

print(len(gisaid_list))

A3_NS
                                                 Header  \
5     EPI_ISL_19490386|A/American_wigeon/Oregon/23-0...   
13    EPI_ISL_19533315|A/Cackling_Goose/BC/AIVPHL-25...   
21    EPI_ISL_19589584|A/duck/Hawaii/033876-004/2024...   
29    EPI_ISL_19589583|A/dove/Hawaii/033876-005/2024...   
37    EPI_ISL_19589586|A/duck/Hawaii/033876-002/2024...   
...                                                 ...   
4330  EPI_ISL_19158241|A/northern_pintail/Alaska/IZ2...   
4345  EPI_ISL_17964877|A/red-tailed_hawk/California/...   
4357  EPI_ISL_19303641|A/Cackling_Goose/BC/AIVPHL-16...   
4365  EPI_ISL_19303630|A/Peregrine_Falcon/BC/AIVPHL-...   
4382  EPI_ISL_18665532|A/Striped_Skunk/BC/AIVPHL-976...   

                  Isolate_Id  \
5     23-032793-004-original   
13               AIVPHL-2533   
21                033876-004   
29                033876-005   
37                033876-002   
...                      ...   
4330               IZ22_0637   
4345           23-006644-002 

### Throw away duplicate isolates from GISAID

In [12]:
gisaid_dict = {}

for gisaid_genotype in gisaid_list: 
    gisaid_genotype_df = gisaid_list[gisaid_genotype]
    print("original:", len(gisaid_genotype_df))
    print(gisaid_genotype)
    if gisaid_genotype in andersen_ncbi_genotypes.keys(): # If they share the genotype
        # Left inner on gisaid, so we can get all duplicates and ignore unique Andersen entries
        gisaid_duplicates = pd.merge(gisaid_genotype_df, andersen_ncbi_genotypes[gisaid_genotype], how="inner") #, indicator=True) #, join="left") #, on=shared_columns)
        # print(gisaid_duplicates)
        gisaid_deduplicated = pd.concat([gisaid_genotype_df, gisaid_duplicates]).drop_duplicates(subset=["Partials"], keep="first") # Use Andersen/NCBI_Virus duplicates instead of GISAID
        gisaid_dict[gisaid_genotype] = gisaid_deduplicated
        print("new:", len(gisaid_deduplicated))
    else: # If this is a GISAID-only genotype
        gisaid_deduplicated = gisaid_genotype_df.drop_duplicates(subset=["Partials"], keep="last")
        gisaid_dict[gisaid_genotype] = gisaid_deduplicated
        print("new:", len(gisaid_deduplicated))
 

original: 514
A3_NS
new: 514
original: 514
A3_HA
new: 514
original: 514
A3_PA
new: 514
original: 514
A3_PB1
new: 514
original: 514
A3_PB2
new: 514
original: 514
A3_NA
new: 514
original: 514
A3_MP
new: 514
original: 514
A3_NP
new: 514


### Create animal reference if needed 

In [13]:
# Find animals to sort, if needed

os.chdir(references)

# Flatten unique_animals
every_unique_animal = []
for l in unique_animals_all:
    for animal in l:
        every_unique_animal.append(animal)

# Rename host type

unique_animals_set = list(set(every_unique_animal))
animals_df = pd.DataFrame(columns=["wild_avian", "domestic_avian", "cattle", "feline", "other_mammal", "human", "pet_food", "other"])
animals_df["other"] = unique_animals_set # to sort

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# print(common_animals)
# print(len(common_animals))

different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print("New animals to add to reference:", different_animals)

print(animals_ref)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    nan_row = pd.DataFrame([[np.nan] * len(animals_df.columns)], columns=animals_df.columns)
    for i in range(number_of_times_to_add_nan):
        animals_df = pd.concat([animals_df, nan_row], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

os.chdir(references)
animals_df.to_csv("animals_ref_to_sort.csv", index=False)

New animals to add to reference: []
            wild_avian domestic_avian               cattle        feline  \
0     great_horned_owl       pheasant            dairy_cow           cat   
1         common_raven         turkey               cattle  domestic_cat   
2        cooper's_hawk        chicken  cattle milk product     feral_cat   
3         coopers_hawk          goose          bovine_milk        feline   
4              peafowl    guinea_fowl              bovine   domestic-cat   
...                ...            ...                  ...           ...   
1663               NaN            NaN                  NaN           NaN   
1664               NaN            NaN                  NaN           NaN   
1665               NaN            NaN                  NaN           NaN   
1666               NaN            NaN                  NaN           NaN   
1667               NaN            NaN                  NaN           NaN   

       other_mammal               human         oth

In [14]:
# Ensure that user checks if there are any new animals

input("Check animals output. Afterwards, press ESCAPE to continue.")

''

## Merge all fastas into different files -- 8 segments * X genotypes ##

In [15]:
# Now separate huge_fasta into genotypes * segments fastas
big_fastas = []

big_fastas = segment_fastas # If no NCBI Virus/Andersen

for gen in genotypes:
    print(gen)
    for seg in unique_segments:
        print(seg)
        print(gisaid_fasta)
        big_fastas.append(gisaid_fasta)

    print(gisaid_fasta[["Genotype"]])



A3
NS
                                                 Header  \
3     EPI_ISL_19490386|A/American_wigeon/Oregon/23-0...   
11    EPI_ISL_19533315|A/Cackling_Goose/BC/AIVPHL-25...   
19    EPI_ISL_19589584|A/duck/Hawaii/033876-004/2024...   
27    EPI_ISL_19589583|A/dove/Hawaii/033876-005/2024...   
35    EPI_ISL_19589586|A/duck/Hawaii/033876-002/2024...   
...                                                 ...   
4344  EPI_ISL_17964877|A/red-tailed_hawk/California/...   
4355  EPI_ISL_19303641|A/Cackling_Goose/BC/AIVPHL-16...   
4363  EPI_ISL_19303630|A/Peregrine_Falcon/BC/AIVPHL-...   
4370  EPI_ISL_19186486|A/glaucous_gull/Alaska/23-025...   
4379  EPI_ISL_18665532|A/Striped_Skunk/BC/AIVPHL-976...   

                  Isolate_Id  \
3     23-032793-004-original   
11               AIVPHL-2533   
19                033876-004   
27                033876-005   
35                033876-002   
...                      ...   
4344           23-006644-002   
4355             AIVPHL-1690 

In [16]:

# Now that we have x fastas, write the files
for fasta in big_fastas:

    # Fix animals
    fasta = fix_animals(fasta, animals_df)

    print("Original length:", len(fasta))
    fasta_dedup_ids = fasta.drop_duplicates(subset="Identifier", keep="last")
    print("New length:", len(fasta))

    fasta_dict = pd.Series(fasta["Sequence"].values,index=fasta.full_header).to_dict()

    # Create fasta file 
    try: 
        output_path = gisaid_files + fasta["Genotype"].values[0].split(" ")[0] + "_" + fasta["Segment"].values[0] + "_" + start_date + "--" + end_date + ".fasta" # Genotype and Segment should all be the same
        output_file = open(output_path, "w")
        for item in fasta_dict.keys():
            # print(item)
            value = fasta_dict[item] + "\n"
            # print(value)
            item = item.replace(" ", "_")
            # print(item)
            output_file.write(item + "\n")
            output_file.write(value)
        print("Succeeded in finding results for genotype: ", fasta["Genotype"].values[0])
        output_file.close()
    except:

        print("Could not find any results for genotype.")

print(len(big_fastas[0]))

Original length: 548
New length: 548
Succeeded in finding results for genotype:  A3
Original length: 548
New length: 548
Succeeded in finding results for genotype:  A3
Original length: 548
New length: 548
Succeeded in finding results for genotype:  A3
Original length: 548
New length: 548
Succeeded in finding results for genotype:  A3
Original length: 548
New length: 548
Succeeded in finding results for genotype:  A3
Original length: 548
New length: 548
Succeeded in finding results for genotype:  A3
Original length: 548
New length: 548
Succeeded in finding results for genotype:  A3
Original length: 548
New length: 548
Succeeded in finding results for genotype:  A3
Original length: 548
New length: 548
Succeeded in finding results for genotype:  A3
Original length: 548
New length: 548
Succeeded in finding results for genotype:  A3
Original length: 548
New length: 548
Succeeded in finding results for genotype:  A3
Original length: 548
New length: 548
Succeeded in finding results for genoty

## Concatenate to Andersen_NCBI files and save

In [17]:

# Concat
os.chdir(complete_files)
segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]

for gisaid_fasta_key in gisaid_dict:
    print(gisaid_fasta_key)
    for andersen_ncbi_fasta_key in andersen_ncbi_genotypes:
        if gisaid_fasta_key == andersen_ncbi_fasta_key: # If the genotypes/segments are the same
            print(andersen_ncbi_fasta_key)
            gisaid_fasta = gisaid_dict[gisaid_fasta_key]
            andersen_ncbi_fasta = andersen_ncbi_genotypes[andersen_ncbi_fasta_key]
            print(andersen_ncbi_fasta.columns)
            combined_fasta = pd.concat([gisaid_fasta, andersen_ncbi_fasta], ignore_index=True).drop_duplicates(subset=["Partials", "Year"], keep="last")
            combined_fasta = combined_fasta.rename(columns={"Sequence":"sequence"})

            combined_fasta["full_header"] = combined_fasta["full_header"].apply(lambda x: ">" + x if ">" not in x else x)

            print(combined_fasta) 
            df_to_fasta(combined_fasta, gisaid_fasta_key + "_" + date_range + ".fasta", Path(complete_files))



A3_NS
A3_NS
Index(['Header', 'Isolate_Id', 'Isolate_Name', 'Subtype', 'Partials',
       'Location', 'Geo_Location', 'Date Collected', 'Species', 'Host_Type',
       'Genotype', 'Sequence', 'Identifier', 'full_header', 'Year', 'Segment'],
      dtype='object')
                                                Header  \
0    EPI_ISL_19490386|A/American_wigeon/Oregon/23-0...   
1    EPI_ISL_19533315|A/Cackling_Goose/BC/AIVPHL-25...   
67   EPI_ISL_19186462|A/short-tailed_shearwaters/Mi...   
76   EPI_ISL_19660998|A/great_horned_owl/Oregon/23-...   
77   EPI_ISL_19660960|A/American_wigeon/Alaska/23-0...   
..                                                 ...   
931  GCA_039823905.1|A/northern_pintail/USA/IZ22_08...   
932  GCA_039640125.1|A/black-legged_kittiwake/Alask...   
933  GCA_039320585.1|A/Northern_Pintail/USA/IZ22_03...   
934  GCA_039295575.1|A/Bald_Eagle/Alaska/22-013001-...   
935  GCA_039296015.1|A/Bald_Eagle/Alaska/22-013831-...   

                 Isolate_Id  \
0    23-032